# AI 면접 질문 생성 에이전트

이 노트북은 각 Phase 모듈을 직접 import해서 순서대로 실행합니다.


In [10]:
from dotenv import load_dotenv
from openai import OpenAI
from pprint import pprint
import importlib
import json

load_dotenv()

import lib.phase1 as phase1_module
import lib.phase2 as phase2_module
import lib.phase3 as phase3_module
import lib.phase4 as phase4_module
import lib.phase5 as phase5_module
import lib.phase6 as phase6_module
import lib.pipeline as pipeline_module

importlib.reload(phase1_module)
importlib.reload(phase2_module)
importlib.reload(phase3_module)
importlib.reload(phase4_module)
importlib.reload(phase5_module)
importlib.reload(phase6_module)
importlib.reload(pipeline_module)

from lib.phase1 import run_jd_workers
from lib.phase2 import (
    classify_personal_statement_sections,
    analyze_and_match_essay,
    build_question_context,
    aggregate_phase2_results,
)
from lib.phase3 import generate_interview_questions
from lib.phase4 import optimize_questions_with_retries, get_passed_final_questions
from lib.phase5 import phase5, format_phase5_output
from lib.phase6 import generate_answers_from_phase5
from lib.pipeline import default_phase1_llm_async_fn, run_interview_question_pipeline_async

client = OpenAI()


## 입력 데이터 로드


In [11]:
with open('JD-sample.md', 'r', encoding='utf-8') as f:
    jd_text = f.read()

with open('essay-sample.md', 'r', encoding='utf-8') as f:
    essay_text = f.read()

print(jd_text[:500])


# 삼성전자 DS부문 AI센터 SW개발 Job Description

---

## 📌 기본 정보

* **직무**: SW개발
* **근무지**: 경기도 수원, 기흥, 화성

AI 및 S/W 기술에 대한 전문 지식을 바탕으로,
DS부문 반도체 생산 및 공정의 지능화를 위한 AI 시스템을 개발하고,
S/W 개발 품질 향상 및 문화를 개선하는 직무

---

## 🧠 Role

* 반도체 도메인 특화 AI 모델, Agent 시스템/서비스, Platform 연구 및 개발
* 반도체 공정/수율/설비 데이터 기반 AI Agent 아키텍처 연구 및 설계
* LLM 기반 추론 구조 및 멀티 Agent 협업 시스템 개발
* 반도체 분석 Workflow 자동화 Agent 설계 및 고도화
* Tool 연계, RAG 구조 및 도메인 지식 기반 추론 전략 구현
* Agent 성능, 추론 정확도, 응답 지연 및 비용 구조 최적화 연구
* 도메인 전문가 협업을 통한 AI 모델 및 시스템 고도화
* AI Ag


## Phase 1 · JD 멀티워커 분석


In [12]:
phase1_results = await run_jd_workers(jd_text, default_phase1_llm_async_fn)
print(f'phase1 worker count: {len(phase1_results)}')
pprint(phase1_results)


gpt-4o 완료
gpt-4o 완료
gpt-4o 완료
phase1 worker count: 5
[{'agent_id': 'role',
  'company_name': '삼성전자 DS부문 AI센터',
  'job_title': 'SW개발',
  'payload': {'roles': [{'question_type': '직무적합',
                         'required_skills': ['반도체 도메인 지식', 'AI 모델 연구 및 개발'],
                         'role_name': '반도체 도메인 특화 AI 모델, Agent 시스템/서비스, '
                                      'Platform 연구 및 개발'},
                        {'question_type': '의사결정',
                         'required_skills': ['데이터 기반 아키텍처 설계', 'AI Agent 시스템'],
                         'role_name': '반도체 공정/수율/설비 데이터 기반 AI Agent 아키텍처 연구 및 '
                                      '설계'},
                        {'question_type': '기술깊이',
                         'required_skills': ['LLM', '멀티 Agent 시스템'],
                         'role_name': 'LLM 기반 추론 구조 및 멀티 Agent 협업 시스템 개발'},
                        {'question_type': '의사결정',
                         'required_skills': ['Workflow 자동화', 'Agent 설계'],
                         'role_n

## Phase 2 · 자소서 분리 및 JD 매칭


In [13]:
personal_statement_list = classify_personal_statement_sections(essay_text)
print(f'personal statement count: {len(personal_statement_list)}')
pprint(personal_statement_list)


personal statement count: 4
[{'answer': '[40시간의 업무 병목을 해소한 설계 역량, 지연 없는 AI 서빙으로 이어가다]\n'
            '삼성전자 DS부문의 Autonomous Fab이 현장에 안착하려면 데이터를 지연 없이 처리하는 서빙 플랫폼의 성능이 '
            '보장되어야 합니다. 수많은 설비에서 발생하는 데이터를 실시간으로 분석하고 결과를 반환하는 과정에서 응답 지연이 '
            '발생하면 제조 공정 전체의 병목으로 이어집니다. 저는 삼성전자 네트워크사업부 연계 프로젝트에서 레거시 엑셀 중심으로 '
            '진행되던 업무를 웹 기반 협업 시스템으로 재설계해 40시간의 작업 시간을 1시간으로 단축한 경험이 있습니다. 현장 '
            '실무자의 업무 병목을 해결한 역량을 바탕으로 DS부문의 AI 플랫폼 안정성을 책임지기 위해 지원했습니다.\n'
            '\n'
            '입사 후 목표는 현장 엔지니어들이 지연 없이 사용할 수 있는 AI Agent 백엔드 환경을 구축하는 것입니다. 이를 '
            '위해 두 단계의 기술적 로드맵을 실행하겠습니다. 첫째, 반도체 공정에서 발생하는 방대한 비정형 데이터의 특성을 '
            '분석하고, 이에 최적화된 데이터베이스 I/O 통제 및 인덱스 설계 기준을 수립하겠습니다. 둘째, 무거운 AI 연산 '
            '결과가 대규모 트래픽 속에서도 안정적으로 반환될 수 있도록 캐싱 전략과 아키텍처를 단계적으로 고도화하겠습니다. '
            '지속적인 트래픽 모니터링과 쿼리 튜닝을 통해 데이터 병목을 사전에 차단하고 견고한 실시간 서빙 인프라를 완성에 기여 '
            '하겠습니다.',
  'question': '삼성전자를 지원한 이유와 입사 후 회사에서 이루고 싶은 꿈을 기술하십시오'},
 {'answer': '[통제 가능한 루틴으로 불확실

In [14]:
phase2_analysis = analyze_and_match_essay(
    js_list=phase1_results,
    personal_statement_list=personal_statement_list,
    client=client,
    model='gpt-4o-mini',
)
pprint(phase2_analysis)


{'items': [{'answer_text': '[40시간의 업무 병목을 해소한 설계 역량, 지연 없는 AI 서빙으로 이어가다]\n'
                           '삼성전자 DS부문의 Autonomous Fab이 현장에 안착하려면 데이터를 지연 없이 '
                           '처리하는 서빙 플랫폼의 성능이 보장되어야 합니다. 수많은 설비에서 발생하는 데이터를 '
                           '실시간으로 분석하고 결과를 반환하는 과정에서 응답 지연이 발생하면 제조 공정 전체의 '
                           '병목으로 이어집니다. 저는 삼성전자 네트워크사업부 연계 프로젝트에서 레거시 엑셀 중심으로 '
                           '진행되던 업무를 웹 기반 협업 시스템으로 재설계해 40시간의 작업 시간을 1시간으로 단축한 '
                           '경험이 있습니다. 현장 실무자의 업무 병목을 해결한 역량을 바탕으로 DS부문의 AI 플랫폼 '
                           '안정성을 책임지기 위해 지원했습니다.\n'
                           '\n'
                           '입사 후 목표는 현장 엔지니어들이 지연 없이 사용할 수 있는 AI Agent 백엔드 환경을 '
                           '구축하는 것입니다. 이를 위해 두 단계의 기술적 로드맵을 실행하겠습니다. 첫째, 반도체 '
                           '공정에서 발생하는 방대한 비정형 데이터의 특성을 분석하고, 이에 최적화된 데이터베이스 '
                           'I/O 통제 및 인덱스 설계 기준을 수립하겠습니다. 둘째, 무거운 AI 연산 결과가 대규모 '
                           '트래픽 속에서도 안정적으로 반환될 

In [15]:
phase2_context = build_question_context(
    analyzed_result=phase2_analysis,
    client=client,
    model='gpt-4o-mini',
)
pprint(phase2_context)


{'items': [{'answer_text': '[40시간의 업무 병목을 해소한 설계 역량, 지연 없는 AI 서빙으로 이어가다]\n'
                           '삼성전자 DS부문의 Autonomous Fab이 현장에 안착하려면 데이터를 지연 없이 '
                           '처리하는 서빙 플랫폼의 성능이 보장되어야 합니다. 수많은 설비에서 발생하는 데이터를 '
                           '실시간으로 분석하고 결과를 반환하는 과정에서 응답 지연이 발생하면 제조 공정 전체의 '
                           '병목으로 이어집니다. 저는 삼성전자 네트워크사업부 연계 프로젝트에서 레거시 엑셀 중심으로 '
                           '진행되던 업무를 웹 기반 협업 시스템으로 재설계해 40시간의 작업 시간을 1시간으로 단축한 '
                           '경험이 있습니다. 현장 실무자의 업무 병목을 해결한 역량을 바탕으로 DS부문의 AI 플랫폼 '
                           '안정성을 책임지기 위해 지원했습니다.\n'
                           '\n'
                           '입사 후 목표는 현장 엔지니어들이 지연 없이 사용할 수 있는 AI Agent 백엔드 환경을 '
                           '구축하는 것입니다. 이를 위해 두 단계의 기술적 로드맵을 실행하겠습니다. 첫째, 반도체 '
                           '공정에서 발생하는 방대한 비정형 데이터의 특성을 분석하고, 이에 최적화된 데이터베이스 '
                           'I/O 통제 및 인덱스 설계 기준을 수립하겠습니다. 둘째, 무거운 AI 연산 결과가 대규모 '
                           '트래픽 속에서도 안정적으로 반환될 

In [16]:
phase2_result = aggregate_phase2_results(
    js_list=phase1_results,
    contextualized_result=phase2_context,
    client=client,
    model='gpt-4o-mini',
)
pprint(phase2_result)


{'global_context': {'global_risks': ['답변이 다소 기술 중심으로 흐를 수 있어 기업 문화와의 맞지 않을 우려가 '
                                     '있음.',
                                     '실제 데이터 분석 경험이 부족할 경우 신뢰성에 의문을 제기할 수 있음.',
                                     'AI Agent 구축 관련 세부 사항에 대한 기술적 깊이가 부족할 '
                                     '가능성.',
                                     '구체적인 AI 기술 경험 부족',
                                     '하드웨어와 소프트웨어의 결합 설명 부족',
                                     '기술적 해결책의 실현 가능성에 대한 의구심',
                                     '아무리 좋은 해결책이라도 반도체 도메인 적용을 명확히 제시하지 않음'],
                    'missing_jd_ids': ['R3',
                                       'R4',
                                       'R6',
                                       'R7',
                                       'R8',
                                       'R9',
                                       'R10',
                                       'R11',
                                       'R13',
   

## Phase 3 · 면접 질문 생성


In [17]:
phase3_result = generate_interview_questions(
    phase2_result,
    client=client,
    orchestrator_model='gpt-4o',
    worker_model='gpt-4o',
)
pprint(phase3_result)


{'questions': ['> 과거 프로젝트에서 AI 시스템을 설계하고 구현한 경험이 있다면, 그 프로젝트에서 본인이 구체적으로 어떤 '
               '역할을 했는지 설명해 주시겠습니까? AI 시스템의 어떤 부분을 주로 담당하셨나요?',
               '> AI 시스템 구현에 필요한 기술들을 어떻게 선택하셨고, 그 기술들의 작동 원리에 대해 간단히 설명해 주실 수 '
               '있나요?',
               '> 프로젝트 중 예상치 못한 문제가 발생했을 때 이를 어떻게 해결하셨는지, 그리고 그 과정에서 어떤 교훈을 '
               '얻으셨는지 말씀해 주시겠습니까?',
               '> 삼성전자 DS부문에서 AI 시스템을 활용하여 어떻게 비즈니스 가치를 높일 수 있을까요? 현재의 AI 기술 '
               '트렌드를 고려하여 제안해 주세요.',
               '반도체 데이터 최적화와 관련하여 진행했던 프로젝트의 구체적인 역할과 기여도를 설명해 주세요.',
               'AI 플랫폼 안정성을 높이기 위해 사용하신 기술 중 가장 효과적이었다고 생각하는 것은 무엇이며, 그 이유는 '
               '무엇인가요?',
               '프로젝트 진행 중 예상치 못한 문제가 발생했을 때 어떻게 대응하셨는지 사례를 공유해 주세요.',
               '삼성전자 DS부문 AI센터의 비즈니스 목표에 맞춰 당신의 역량을 어떻게 적용할 수 있을지 설명해 주시겠어요?',
               '이전에 수행한 AI 플랫폼 안정성 관련 프로젝트에서 당신의 역할과 기여는 무엇이었나요?',
               'AI 시스템 설계 시 특정 기술이나 방법론을 선택한 이유와 그 작동 원리를 설명해 주세요.',
               '예상치 못한 문제나 실패 사례가 있었다면 어떻게 대처하셨나요?',
             

## Phase 4 · 질문 평가 및 개선


In [18]:
job_category = phase2_result.get('target', {}).get('role', '백엔드개발자')

phase4_results = optimize_questions_with_retries(
    phase3_result['questions'],
    job_category,
    max_retries=3,
)
phase4_pass_questions = get_passed_final_questions(phase4_results)

print('passed question count:', len(phase4_pass_questions))
pprint(phase4_results)


passed question count: 20
[{'attempts': [],
  'final_evaluation': '평가 결과: PASS\n'
                      '\n'
                      '문제점 및 개선 방향:\n'
                      '- 없음\n'
                      '\n'
                      '이 질문은 면접에서 지원자의 경험을 심도 있게 탐구할 수 있는 훌륭한 질문입니다. 꼬리질문으로 AI '
                      '시스템의 특정 기술 스택, 팀 내 협업 및 문제 해결 과정에 대한 질문으로 이어질 수 있으며, '
                      '지원자가 구체적이고 관련된 경험을 기반으로 답변할 수 있도록 하여 JD에서 요구하는 역량과도 '
                      '연결됩니다. 자기소개서에서의 경험을 토대로 확장할 수 있는 기회도 제공합니다. 또한, 실제 면접관이 '
                      '활용할 만한 적절한 질문입니다.',
  'final_question': '> 과거 프로젝트에서 AI 시스템을 설계하고 구현한 경험이 있다면, 그 프로젝트에서 본인이 구체적으로 '
                    '어떤 역할을 했는지 설명해 주시겠습니까? AI 시스템의 어떤 부분을 주로 담당하셨나요?',
  'final_status': 'PASS',
  'initial_evaluation': '평가 결과: PASS\n'
                        '\n'
                        '문제점 및 개선 방향:\n'
                        '- 없음\n'
                        '\n'
                        '이 질문은 면접에서 지원자의 경험을 심도 있게 탐구할 수 있는 훌륭한 질문입니다. 꼬리질문으로 '
    

## Phase 5 · 질문 분류 및 우선순위 리포트


In [19]:
phase5_result = phase5(phase4_pass_questions, job_category)
phase5_report = format_phase5_output(phase5_result)

print(phase5_report)


🏆 직무별 핵심 예상 질문 리포트

■ 프로젝트 경험 검증
  1위. 과거 프로젝트에서 AI 시스템을 설계하고 구현한 경험이 있다면, 그 프로젝트에서 본인이 구체적으로 어떤 역할을 했는지 설명해 주시겠습니까? AI 시스템의 어떤 부분을 주로 담당하셨나요?
  2위. 이전에 수행한 AI 플랫폼 안정성 관련 프로젝트에서 당신의 역할과 기여는 무엇이었나요?
  3위. 자기소개서에서 언급한 AI 에이전트 백엔드 프로젝트에서 사용한 프로그래밍 언어와 프레임워크는 무엇이며, 이를 통해 팀에 기여한 구체적인 성과나 경험에 대해 설명해 주세요.
  4위. 과거에 AI 플랫폼 구축 중 예상치 못한 변수가 발생했을 때 어떻게 대처했는지 말씀해 주세요.
  5위. 최근 참여했던 자연어 처리 또는 AI 프로젝트 중 하나를 선택해 고객 경험을 개선한 구체적인 방법과 그 과정에서 팀워크나 문제 해결 능력을 어떻게 발휘했는지 설명해 주세요.
  6위. 자기소개서에 언급한 반도체 데이터 최적화 프로젝트에서 어떤 역할을 맡았고, 이 과정에서 직면한 기술적 문제는 무엇이었으며, 이를 어떻게 해결했는지 구체적으로 설명해 주세요.
  7위. 웹 기반 협업 시스템 재설계 과정에서 예상치 못한 어려움이 있었나요? 이를 어떻게 극복했는지 설명해주세요.
■ AI/LLM/Agent 역량 검증
  1위. AI 시스템 구현에 필요한 기술들을 어떻게 선택하셨고, 그 기술들의 작동 원리에 대해 간단히 설명해 주실 수 있나요?
  2위. 삼성전자 DS부문에서 AI 시스템을 활용하여 어떻게 비즈니스 가치를 높일 수 있을까요? 현재의 AI 기술 트렌드를 고려하여 제안해 주세요.
  3위. 삼성전자 DS부문 AI센터의 데이터 활용 극대화 및 AI 솔루션 개발 목표에 대해, 당신의 데이터 처리 및 AI 모델 개발 경험을 활용하여 어떤 기여를 할 수 있을지 구체적인 사례와 함께 설명해 주시겠습니까?
  4위. 당신의 경력 중 머신러닝 기술을 활용한 경험을 바탕으로, 삼성전자 DS부문에서 제품 개선이나 프로세스 혁신을 위한

## Phase 6 · Phase 5 질문 기반 모범 답변 생성


In [20]:
phase6_result = generate_answers_from_phase5(
    phase5_result,
    job_category,
    top_n_per_category=2,
)

pprint(phase6_result)


{'AI/LLM/Agent 역량 검증': {'answers': [{'difficulty': 4,
                                     'frequency': 3,
                                     'question': 'AI 시스템 구현에 필요한 기술들을 어떻게 '
                                                 '선택하셨고, 그 기술들의 작동 원리에 대해 간단히 '
                                                 '설명해 주실 수 있나요?',
                                     'rank': 1,
                                     'reason': 'AI 기술의 선택과 작동 원리에 대한 깊은 이해 필요.',
                                     'relevance': 5,
                                     'sample_answer': '**상황(S)**: 지난 프로젝트에서 '
                                                      '고객의 요구를 반영한 AI 챗봇 시스템을 '
                                                      '구현하는 일이 주어졌습니다. 이 시스템은 '
                                                      '고객 응대의 효율성을 높이기 위해 필요한 '
                                                      '기술 선택이 중요했습니다.\n'
                                                      '\n'
                                       

## End-to-End 실행

위 단계를 한 번에 실행하고 싶으면 파이프라인 함수를 사용하면 됩니다.


In [ ]:
pipeline_result = await run_interview_question_pipeline_async(
    jd_text=jd_text,
    essay_text=essay_text,
    client=client,
)

print(pipeline_result['phase5_report'])


gpt-4o 완료
gpt-4o 완료
gpt-4o 완료
